## Data Ingestion

In [44]:
#Core Libraries
import pandas as pd
import polars as pl
import numpy as np

# Scikit-learn: Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

#Scikit-learn: Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

#Scikit-learn: Metrics
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report,
    make_scorer, accuracy_score, f1_score, roc_auc_score

)

# Scikit-learn: Model Selection & Evaluation
from sklearn.model_selection import (
    train_test_split,
    cross_validate
)

In [17]:
#Loading in the cleaned data
try:
    taxi_trip_df = pl.read_parquet("data/cleaned-taxi-data.parquet")
    taxi_zone_df = pl.read_csv("data/taxi_zone_lookup.csv")
except Exception as e:
    print("Error: ", e)

#Filtering to only use rows with credit card payment
taxi_trip_df = taxi_trip_df.filter(pl.col("payment_type") == 1)

print("Taxi Data Df")
taxi_trip_df.head()

Taxi Data Df


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,"""N""",148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0


In [18]:
print("Taxi Zone Df")
taxi_zone_df.head()

Taxi Zone Df


LocationID,Borough,Zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""


## Part 1: Data Preprocessing & Feature Engineering 

### Feature Engineering

In [19]:
#Creating Temporal Features

#Adding Pickup Hour
taxi_trip_df = taxi_trip_df.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour')
])

#Adding Pick Day of Week (Integer Form)
taxi_trip_df = taxi_trip_df.with_columns([
    pl.col('tpep_pickup_datetime').dt.weekday().alias('pickup_day_of_week')
])

#Adding Is Weekend Boolean Column
taxi_trip_df = taxi_trip_df.with_columns([
    pl.when(pl.col('pickup_day_of_week') >= 5)
    .then(True)
    .otherwise(False)
    .alias("is_weekend")
])


In [20]:
#Creating Trip Features

#Adding trip duration in minutes
taxi_trip_df = taxi_trip_df.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
     .dt.total_seconds() / 60)
     .alias("trip_duration_minutes")
])

#Adding trip speed in mph
taxi_trip_df = taxi_trip_df.with_columns([
    pl.when(pl.col("trip_duration_minutes") > 0)
      .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
      .otherwise(None)
      .alias("trip_speed_mph")
])

#Adding log trip distance (log transformed distance)
taxi_trip_df = taxi_trip_df.with_columns([
    pl.col('trip_distance').log1p().alias('log_trip_distance')
])



In [21]:
#Creating Fare Features

#Adding fare per mile column
taxi_trip_df = taxi_trip_df.with_columns([
    pl.when(pl.col('trip_distance') > 0)
    .then( pl.col('fare_amount') / pl.col('trip_distance') )
    .otherwise(None)
    .alias('fare_per_mile')
])

#Adding fare per minute column
taxi_trip_df = taxi_trip_df.with_columns([
    pl.when(pl.col('trip_duration_minutes') > 0)
    .then(pl.col('fare_amount') / pl.col('trip_duration_minutes'))
    .otherwise(None)
    .alias('fare_per_minute')
])

In [22]:
#Creating Zone Features

#Joining Taxi Zone and Taxi Data Dataframe on Location ID

#Adding Pickup Borough 
taxi_trip_df =taxi_trip_df.join(
    taxi_zone_df.select(["LocationID","Borough"]),
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
).rename({"Borough":"pickup_borough"})

#Adding Dropoff Borough 
taxi_trip_df =taxi_trip_df.join(
    taxi_zone_df.select(["LocationID","Borough"]),
    left_on="DOLocationID",
    right_on="LocationID",
    how="left"
).rename({"Borough":"dropoff_borough"})


In [23]:
#Creating OneHotEncoder to encode Pickup and Dropoff Burough
#Sprase output set to false to return dense numpy array, it will convert easier to Polars df
encoder = OneHotEncoder(sparse_output=False)

#Encode the two columns and convert to numpy array
burough_columns = encoder.fit_transform(
    taxi_trip_df.select(["pickup_borough", "dropoff_borough"]).to_numpy()
)

#Extract the feature names
feature_names = encoder.get_feature_names_out(
    ["pickup_borough", "dropoff_borough"]
)

#Convert back into a polars dataframe
encoded_df = pl.DataFrame(burough_columns, schema=feature_names.tolist())

#Concat with original df but drop old columns
taxi_trip_df = pl.concat([
    taxi_trip_df.drop(["pickup_borough", "dropoff_borough"]),
    encoded_df,
], how="horizontal")


### Target Variable Creation

In [24]:
#Creating high_tip variable
taxi_trip_df = taxi_trip_df.with_columns([
    pl.when(pl.col('fare_amount') > 0)
    .then((pl.col('tip_amount') > 0.2 * pl.col('fare_amount')).cast(pl.Int8))
    .otherwise(0)
    .alias('high_tip')
])

#Storing the tip_amount variable and converting to numpy array
y_tip_amount = taxi_trip_df['tip_amount']
y_regression = y_tip_amount.to_numpy()

#Storing high_tip variable and converting to numpy array
y_high_tip = taxi_trip_df['high_tip']
y_classification = y_high_tip.to_numpy()

#Droping target variables and unwanted variables and storing new dataset for training
X = taxi_trip_df.drop([
    'tip_amount',
    'high_tip',
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'store_and_fwd_flag'
])

### Data Splitting & Scaling

In [25]:
#Splitting the data for training

#Initial split to get train data split
X_train, X_temp, y_regression_train, y_regression_temp, y_classification_train, y_classification_temp = train_test_split(
    X, y_regression, y_classification, test_size=0.3, random_state=42, stratify=y_classification
)

#Second split to get test and validation data
X_val, X_test, y_regression_val, y_regression_test, y_classification_val, y_classification_test = train_test_split(
    X_temp, y_regression_temp, y_classification_temp, test_size=0.5, random_state=42, stratify=y_classification_temp
)


In [26]:
#Applying appropriate scaling to training data

#Convert to pandas to do transformation of data
X_train = X_train.to_pandas()
X_val = X_val.to_pandas()
X_test = X_test.to_pandas()

#Extracting numeric features and categorical features
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

#Creating a column transformer to do the scaling
numeric_transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ],
    remainder='passthrough' 
)

#Fitting to training data so that only this data learns the parameters
numeric_transformer.fit(X_train)

#Transforming all data to apply same parameters
X_train_scaled = numeric_transformer.transform(X_train)
X_test_scaled = numeric_transformer.transform(X_test)
X_val_scaled = numeric_transformer.transform(X_val)


#Converting to Pandas df for Modelling from here on out 
feature_names = numeric_transformer.get_feature_names_out()

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_names)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)

In [27]:
# Check split sizes to confirm correct splitting
print(f"Training set: {len(X_train_scaled):,} samples ({len(X_train_scaled)/len(X)*100:.1f}%)")
print(f"Validation set: {len(X_val_scaled):,} samples ({len(X_val_scaled)/len(X)*100:.1f}%)")
print(f"Test set: {len(X_test_scaled):,} samples ({len(X_test_scaled)/len(X)*100:.1f}%)")
print(f"Total: {len(X_train_scaled) + len(X_val_scaled) + len(X_test_scaled):,} samples (should equal {len(X):,})")

#Getting class distributions
print(f'\nClass distribution for classification variable (high tip):')
print(f'\nTraining class distribution:')
print(pd.Series(y_classification_train).value_counts(normalize=True).round(4))
print(f'\nTest class distribution:')
print(pd.Series(y_classification_test).value_counts(normalize=True).round(4))
print(f'\nValidation class distribution:')
print(pd.Series(y_classification_val).value_counts(normalize=True).round(4))

Training set: 1,608,888 samples (70.0%)
Validation set: 344,762 samples (15.0%)
Test set: 344,762 samples (15.0%)
Total: 2,298,412 samples (should equal 2,298,412)

Class distribution for classification variable (high tip):

Training class distribution:
1    0.7593
0    0.2407
Name: proportion, dtype: float64

Test class distribution:
1    0.7593
0    0.2407
Name: proportion, dtype: float64

Validation class distribution:
1    0.7593
0    0.2407
Name: proportion, dtype: float64


In [28]:
#Feature name summary, types and features excluded

#Arrange feature summary into table for better readablity
feature_summary = pd.DataFrame({
    "Feature": list(X.schema.keys()),
    "Type": list(X.schema.values())
})

print("\nFeature Summary")
display(feature_summary)


print("\nExcluded Features: tip_amount and high_tip")
print("\nThese are our target variables for our modelling so they were dropped from the dataset")
print("and will be used by our model for classification(high_tip) and regression(tip_amount).")
print("Futhermore they were excluded to prevent data leakage.")
print("\I also dropped any Date related and String related fields for model training.")


Feature Summary


,Feature,Type
0,VendorID,Int32
1,passenger_count,Int64
2,trip_distance,Float64
3,RatecodeID,Int64
4,PULocationID,Int32
5,DOLocationID,Int32
6,payment_type,Int64
7,fare_amount,Float64
8,extra,Float64
9,mta_tax,Float64



Excluded Features: tip_amount and high_tip

These are our target variables for our modelling so they were dropped from the dataset
and will be used by our model for classification(high_tip) and regression(tip_amount).
Futhermore they were excluded to prevent data leakage.
\I also dropped any Date related and String related fields for model training.


## Part 2: Model Training & Tuning

### Baseline Models 

#### Training Regression Models (Linear Regression and Random Forest Regressor)

In [29]:
#Initializing models
linearRegressionModel = LinearRegression()
randForestRegressModel = RandomForestRegressor(n_estimators=100, random_state=42)

#Removing Nans
X_train_scaled = X_train_scaled.fillna(X_train_scaled.median())
X_val_scaled = X_val_scaled.fillna(X_val_scaled.median())
X_test_scaled = X_test_scaled.fillna(X_test_scaled.median())

#Training models on data
print(f'Training Linear Regression model...')
linearRegressionModel.fit(X_train_scaled, y_regression_train)
print("    ✓ Linear Regression trained")

print(f'Training Random Forest Regression model...')
randForestRegressModel.fit(X_train_scaled, y_regression_train)
print("    ✓ Random Forest Regression trained")


C:\Users\arves\AppData\Local\Temp\ipykernel_15812\2431582643.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_scaled = X_train_scaled.fillna(X_train_scaled.median())
C:\Users\arves\AppData\Local\Temp\ipykernel_15812\2431582643.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_val_scaled = X_val_scaled.fillna(X_val_scaled.median())
C:\Users\arves\AppData\Local\Temp\ipykernel_15812\2431582643.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_ob

Training Linear Regression model...
    ✓ Linear Regression trained
Training Random Forest Regression model...
    ✓ Random Forest Regression trained


#### Training Classification Models (Logistic Regression and a Random Forest Classifier)

In [30]:
#Initializing models
logisticRegressioModel = LogisticRegression(max_iter=1000, random_state=42)
randForestClassifierModel = RandomForestClassifier(n_estimators=100, random_state=42,n_jobs=-1)

#Training models on data
print(f'Training Logistic Regression model...')
logisticRegressioModel.fit(X_train_scaled, y_classification_train)
print("    ✓ Logistic Regression trained")

print(f'Training Random Forest Classifier model...')
randForestClassifierModel.fit(X_train_scaled, y_classification_train)
print("    ✓ Random Forest Classifier trained")

Training Logistic Regression model...


C:\Users\arves\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


    ✓ Logistic Regression trained
Training Random Forest Classifier model...
    ✓ Random Forest Classifier trained


#### Reporting Performance of Models on Validation Data (Regression)

In [35]:
# Storing Regression Models
regressionModels = {
    "Linear Regression": linearRegressionModel,
    "Random Forest Regressor": randForestRegressModel
}

regression_results = {}

# Training and Validation Evaluation
for name, model in regressionModels.items():
    # Predict on validation set
    preds = model.predict(X_val_scaled)

    # Calculate metrics
    mae = mean_absolute_error(y_regression_val, preds)
    rmse = np.sqrt(mean_squared_error(y_regression_val, preds))
    r2 = r2_score(y_regression_val, preds)

    regression_results[name] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

# Output results
print('\n' + '=' * 60)
print(f'{"Model":>25s} | {"MAE":>5s} | {"RMSE":>5s} | {"R2":>5s}')
print('-' * 60)

for name, r in regression_results.items():
    print(f'{name:>25s} | {r["MAE"]:.4f} | {r["RMSE"]:.4f} | {r["R2"]:.4f}')


                    Model |   MAE |  RMSE |    R2
------------------------------------------------------------
        Linear Regression | 0.0752 | 0.2218 | 0.9967
  Random Forest Regressor | 0.0443 | 0.5264 | 0.9813


#### Reporting Performance of Models on Validation Data (Classification)

In [ ]:
#Storing Regression Models in Dictionary
classificationModels = {
    "Logistic Regression": logisticRegressioModel,
    "Random Forest Classifier": randForestClassifierModel
}


classification_results = {}

# Training and Validation Evaluation
for name, model in classificationModels.items():
    
    # Predict on validation set
    preds = model.predict(X_val_scaled)

    # Get prediction probabilities for AUC-ROC
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    # Calculate classification metrics
    accuracy = accuracy_score(y_classification_val, preds)
    precision = precision_score(y_classification_val, preds, zero_division=0)
    recall = recall_score(y_classification_val, preds, zero_division=0)
    f1 = f1_score(y_classification_val, preds, zero_division=0)
    auc_roc = roc_auc_score(y_classification_val, y_pred_proba)

    classification_results[name] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "AUC-ROC": auc_roc
    }

# Output results
print('\n' + '=' * 60)
print(f'{"Model":>25s} | {"Accuracy"} | {"Precision"} | {"Recall"} | {"F1-Score"} | {"AUC-ROC"}')
print('-' * 60)

for name, r in classification_results.items():
    print(f'{name:>25s} | {r["Accuracy"]:.4f} | {r["Precision"]:.4f} | {r["Recall"]:.4f} | {r["F1-Score"]:.4f} | {r["AUC-ROC"]:.4f}')



                    Model | Accuracy | Precision | Recall | F1-Score | AUC-ROC
------------------------------------------------------------
      Logistic Regression | 0.9908 | 0.9909 | 0.9970 | 0.9940 | 0.9966
 Random Forest Classifier | 0.9823 | 0.9791 | 0.9980 | 0.9885 | 0.9988


### Hyperparameter Tuning

In [33]:
#Sample size chosen to optimize efficiency
SAMPLE_SIZE = 200_000

# Stratified sample to preserve class balance
sample_idx = (
    pd.Series(y_classification_train)
    .groupby(pd.Series(y_classification_train))
    .apply(lambda x: x.sample(frac=SAMPLE_SIZE / len(y_classification_train), random_state=42))
    .index.get_level_values(1)
)

X_tune = X_train_scaled.iloc[sample_idx].reset_index(drop=True)

#REPLACE ACCORDING TO BEST MODEL
y_tune = y_classification_train[sample_idx]

In [ ]:
from scipy.stats import randint, uniform

#Hyper parameters to be tuned
param_distributions = {
    'n_estimators':      randint(100, 200),
    'max_depth':         [10, 20, 30, None],
    'min_samples_split': randint(2, 11),
}

scorings = {
    "Accuracy":  make_scorer(accuracy_score),
    "Precision": make_scorer(precision_score, zero_division=0),
    "Recall":    make_scorer(recall_score, zero_division=0),
    "F1-Score":  make_scorer(f1_score, zero_division=0),
    "AUC-ROC":   make_scorer(roc_auc_score, needs_proba=True)
}

#K Folds Cross Validation
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#REPLACE WITH OPTIMAL MODEL
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1), param_distributions, n_iter=20,
    scoring=scorings, refit="F1-Score", cv=cv_strategy, n_jobs=-1, random_state=42, verbose=1
)

print("Running RandomizedSearchCV (n_iter=20, 5-fold CV)...")
random_search.fit(X_tune, y_tune)
print("\n✓ RandomizedSearchCV completed.\n")

print("Best hyperparameters found:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest cross-validation F1-Score (on sample): {random_search.best_score_:.4f}")

best_index = random_search.best_index_
print("\nOther metrics for the best model:")
for metric in scorings.keys():
    mean_score = random_search.cv_results_[f"mean_test_{metric}"][best_index]
    print(f"  {metric}: {mean_score:.4f}")

Running RandomizedSearchCV (n_iter=20, 5-fold CV)...
Fitting 5 folds for each of 5 candidates, totalling 25 fits


KeyboardInterrupt: 

In [ ]:
#Retraining based on Tuned Parameters
best_model = RandomForestClassifier(
    **random_search.best_params_,
    random_state=42,
    n_jobs=-1
)

print("\nRetraining best model on full training set...")
best_model.fit(X_train_scaled, y_classification_train)
print("✓ Tuned model trained on full training set\n")

def evaluateClassifier(name, model):
    # Predict on validation set
    preds = model.predict(X_val_scaled)

    # Get prediction probabilities for AUC-ROC
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    # Calculate classification metrics
    return {
        "Model":     name,
        "Accuracy":  accuracy_score(y_classification_val, preds),
        "Precision": precision_score(y_classification_val, preds, zero_division=0),
        "Recall":    recall_score(y_classification_val, preds, zero_division=0),
        "F1-Score":  f1_score(y_classification_val, preds, zero_division=0),
        "AUC-ROC":   roc_auc_score(y_classification_val, y_pred_proba),
    }

baseline = evaluateClassifier("Random Forest (Baseline)", randForestClassifierModel)
tuned = evaluateClassifier("Random Forest (Tuned)", best_model)

#Convert to df for better visualization
comparison_df = pd.DataFrame([baseline, tuned]).set_index("Model")

print("Validation Set — Baseline vs Tuned Random Forest Classifier")
print(comparison_df.round(4).to_string())
print()

print("Improvement over baseline:")
for metric in ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"]:
    delta = tuned[metric] - baseline[metric]
    judgement = "Improved" if delta > 0 else ("Worse" if delta < 0 else "Same")
    print(f"  {metric:12s}: {judgement} {delta:+.4f}")

### Neural Network Model

In [ ]:
#Code to Save Models

import joblib
import os

os.makedirs('models', exist_ok=True)

joblib.dump(linearRegressionModel,      'models/linear_regression.joblib')
joblib.dump(randForestRegressModel,     'models/rf_regressor.joblib')
joblib.dump(logisticRegressioModel,     'models/logistic_regression.joblib')
joblib.dump(randForestClassifierModel,  'models/rf_classifier_baseline.joblib')
# joblib.dump(best_model,                 'models/rf_classifier_tuned.joblib')

print("✓ All models saved to models/")

In [ ]:
#Code to Load Models

import joblib

linearRegressionModel     = joblib.load('models/linear_regression.joblib')
randForestRegressModel    = joblib.load('models/rf_regressor.joblib')
logisticRegressioModel    = joblib.load('models/logistic_regression.joblib')
randForestClassifierModel = joblib.load('models/rf_classifier_baseline.joblib')
best_model                = joblib.load('models/rf_classifier_tuned.joblib')

print("✓ All models loaded")